# MachineLearning 项目上传到 GitHub 私有仓库并同步（完整模板）

这个 Notebook 是给你自己长期复用的流程手册：
- 第一次把本地项目上传到 GitHub 私有仓库
- 多台电脑之间同步
- 本地改动后自动推送（定时）
- 常见问题排查


## 0. 先决条件

1. 已安装 Git
2. 已有 GitHub 账号
3. 目标目录：`/home/levono/MachineLearning`
4. 建议使用 SSH（更稳定，免重复输密码）


In [ ]:
# 检查 git
git --version


## 1. 在 GitHub 新建私有仓库

在 GitHub 网页端操作：
1. 点击 New repository
2. Repository name: 建议 `MachineLearning`
3. 选择 `Private`
4. 不要勾选 README / .gitignore / license（避免和本地初始化冲突）
5. 创建后复制仓库地址（SSH 形如 `git@github.com:用户名/MachineLearning.git`）


## 2. 配置 SSH（推荐，一次配置长期可用）


In [ ]:
# 生成 SSH 密钥（如果你已有 ~/.ssh/id_ed25519，可以跳过）
ssh-keygen -t ed25519 -C "your_email@example.com"


In [ ]:
# 启动 agent 并添加私钥
eval "$(ssh-agent -s)"
ssh-add ~/.ssh/id_ed25519


In [ ]:
# 复制公钥，粘贴到 GitHub -> Settings -> SSH and GPG keys
cat ~/.ssh/id_ed25519.pub


In [ ]:
# 测试 SSH 联通
ssh -T git@github.com


成功时会看到类似：`Hi <username>! You've successfully authenticated...`


## 3. 第一次上传本地 MachineLearning


In [ ]:
cd /home/levono/MachineLearning

# 如果还不是仓库
git init

# 设置提交身份（如未设置）
git config --global user.name "你的GitHub用户名"
git config --global user.email "你的GitHub注册邮箱"


### 3.1 建议先写 `.gitignore`

在项目根目录创建 `.gitignore`（避免把缓存、虚拟环境、大模型权重误上传）：


In [ ]:
cat > .gitignore <<'GITIGNORE'
# Python cache
__pycache__/
*.py[cod]
*.so

# Virtual env
.venv/
venv/
env/

# Jupyter
.ipynb_checkpoints/

# OS / IDE
.DS_Store
Thumbs.db
.vscode/
.idea/

# Logs
*.log

# Secrets
.env
*.pem
*.key

# Common ML artifacts (按需保留)
data/
datasets/
checkpoints/
runs/
wandb/
mlruns/

# Large model files
*.pt
*.pth
*.ckpt
*.onnx
*.h5
*.pb
GITIGNORE


In [ ]:
# 首次提交
git add .
git commit -m "init: first upload"

# main 分支
git branch -M main

# 添加远端（把 URL 改成你的）
git remote add origin git@github.com:你的用户名/MachineLearning.git

# 首次推送并建立 upstream
git push -u origin main


## 4. 日常同步（手动，最稳妥）

每次写完代码：


In [ ]:
cd /home/levono/MachineLearning
git add -A
git commit -m "feat/fix: 描述这次改动"
git push


如果远端也有人（或另一台电脑）提交过，先：


In [ ]:
git pull --rebase
git push


## 5. 跨电脑同步标准流程

新电脑上：
1. 配好 Git + SSH
2. `git clone git@github.com:你的用户名/MachineLearning.git`
3. 改代码后照常 `add -> commit -> push`
4. 回到旧电脑先 `git pull --rebase` 再继续开发


In [ ]:
# 新电脑第一次
cd ~
git clone git@github.com:你的用户名/MachineLearning.git
cd MachineLearning


## 6. 自动同步（本地改动后“定时自动推送”）

说明：Git 没有真正“实时自动推送”内建功能。常用做法是 `cron` 每 N 分钟检查并自动提交推送。


In [ ]:
# 在项目目录创建自动同步脚本
cat > /home/levono/MachineLearning/auto-sync.sh <<'SH'
#!/usr/bin/env bash
set -e

cd /home/levono/MachineLearning

# 先同步远端，尽量减少冲突
git pull --rebase --autostash || true

git add -A
if ! git diff --cached --quiet; then
  git commit -m "auto-sync: $(date '+%Y-%m-%d %H:%M:%S')"
  git push
fi
SH

chmod +x /home/levono/MachineLearning/auto-sync.sh


In [ ]:
# 每 5 分钟执行一次
crontab -e

# 在打开的编辑器里添加：
# */5 * * * * /home/levono/MachineLearning/auto-sync.sh >> /tmp/ml-autosync.log 2>&1


In [ ]:
# 查看定时任务是否生效
crontab -l

# 查看日志
tail -n 50 /tmp/ml-autosync.log


## 7. 大文件建议（Git LFS）

如果你要跟踪模型权重（如 `.pt/.ckpt`），建议 Git LFS，否则仓库会变得很慢。


In [ ]:
# 安装后初始化（仅一次）
git lfs install

# 跟踪大文件类型
git lfs track "*.pt"
git lfs track "*.ckpt"

git add .gitattributes
git commit -m "chore: enable git lfs"
git push


## 8. 常见问题排查

### 问题 A：`remote origin already exists`
说明已经有远端配置，改 URL 即可。


In [ ]:
git remote -v
git remote set-url origin git@github.com:你的用户名/MachineLearning.git


### 问题 B：`rejected` / `non-fast-forward`
先拉取再推送：


In [ ]:
git pull --rebase
git push


### 问题 C：合并冲突
1. 打开冲突文件并手动修改
2. `git add 冲突文件`
3. `git rebase --continue`
4. `git push`


### 问题 D：提交了不该提交的文件
把规则写进 `.gitignore` 后，再移出缓存：


In [ ]:
git rm -r --cached .
git add .
git commit -m "chore: apply gitignore"
git push


## 9. 推荐你的日常习惯

1. 每次开始开发前：`git pull --rebase`
2. 改完一小段就提交，commit message 写清楚
3. 不要把数据集和密钥放进仓库
4. 定期在 GitHub 页面看 Commit 历史是否符合预期
